# Piper π0.5 LoRA 학습 계약·1-step smoke notebook

이 notebook은 Piper LeRobot v3 데이터를 OpenPI π0.5 LoRA에 연결하기 위한 **검증용 prototype**이다.
production 학습 코드는 아니다. 여기서 모델·데이터·정규화·LoRA·checkpoint 계약을 먼저 고정한 뒤
`src/piper_vla/training/pi05/`로 옮긴다.

> 중요: OpenPI의 `pi05_libero`는 full fine-tuning 예제이며 Piper LoRA 추천값이 아니다.
> 이 notebook의 optimizer와 LR은 기존 π0 LoRA 안정성 baseline을 사용한 실험 시작점이다.


In [1]:
# Cell 1 — JAX import 전 workspace와 runtime 환경 고정
from pathlib import Path
import os
import sys


def find_workspace_root(start: Path) -> Path:
    """현재 위치의 부모에서 vla_ws root를 찾는다."""

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "third_party" / "openpi").is_dir() and (candidate / "src" / "piper_vla").is_dir():
            return candidate
    raise FileNotFoundError("vla_ws root를 찾지 못했습니다.")


WORKSPACE_ROOT = find_workspace_root(Path.cwd())
OPENPI_ROOT = WORKSPACE_ROOT / "third_party" / "openpi"
WORKSPACE_SOURCE = WORKSPACE_ROOT / "src"
OPENPI_SOURCE = OPENPI_ROOT / "src"

if "jax" in sys.modules:
    raise RuntimeError("JAX가 이미 import됐습니다. Kernel을 재시작한 뒤 Cell 1부터 실행하세요.")

os.environ.setdefault("OPENPI_DATA_HOME", str(WORKSPACE_ROOT / "data" / "cache" / "openpi"))
os.environ.setdefault("HF_HOME", str(WORKSPACE_ROOT / "data" / "cache" / "huggingface"))
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(WORKSPACE_ROOT / "data" / "cache" / "jax"))
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.65")

for source_root in (str(WORKSPACE_SOURCE), str(OPENPI_SOURCE)):
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)

DATASET_ROOT = WORKSPACE_ROOT / "data" / "datasets" / "two_block_pnp"
PI05_BASE_PARAMS = WORKSPACE_ROOT / "data" / "cache" / "openpi" / "openpi-assets" / "checkpoints" / "pi05_base" / "params"
PI0_NORM_PATH = WORKSPACE_ROOT / "data" / "assets" / "pi0_piper_lora" / "two_block_pnp" / "norm_stats.json"
PI05_NORM_PATH = WORKSPACE_ROOT / "data" / "assets" / "pi05_piper_lora" / "two_block_pnp" / "norm_stats.json"
RUNS_ROOT = WORKSPACE_ROOT / "data" / "runs"
PI05_CONFIG_NAME = "pi05_piper_lora"
ASSET_ID = "two_block_pnp"
RUN_NAME = "two_block_pnp_b1_pi05_vf_s1_smoke_r001"
DEFAULT_PROMPT = "pick up the green blocks one at a time and place them in the white box"

print("Workspace :", WORKSPACE_ROOT)
print("Dataset   :", DATASET_ROOT)
print("π0.5 base :", PI05_BASE_PARAMS)
print("Run       :", RUNS_ROOT / PI05_CONFIG_NAME / RUN_NAME)


Workspace : /home/pc/vla_ws
Dataset   : /home/pc/vla_ws/data/datasets/two_block_pnp
π0.5 base : /home/pc/vla_ws/data/cache/openpi/openpi-assets/checkpoints/pi05_base/params
Run       : /home/pc/vla_ws/data/runs/pi05_piper_lora/two_block_pnp_b1_pi05_vf_s1_smoke_r001


## 2. 파일·데이터 사전 검사

이 단계는 JAX와 GPU를 사용하지 않는다. π0.5 base checkpoint, Piper dataset과 기존 raw 통계 파일만 확인한다.


In [2]:
# Cell 2 — GPU를 건드리지 않는 경로·dataset metadata 검사
import json


required_paths = {
    "OpenPI source": OPENPI_SOURCE,
    "Piper dataset": DATASET_ROOT,
    "π0.5 base params": PI05_BASE_PARAMS,
    "source norm stats": PI0_NORM_PATH,
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("필수 경로가 없습니다:\n" + "\n".join(missing))

info = json.loads((DATASET_ROOT / "meta" / "info.json").read_text(encoding="utf-8"))
assert info["codebase_version"] == "v3.0"
assert info["fps"] == 20
assert info["total_episodes"] == 1000
assert info["total_frames"] == 699_921

print("Episodes  :", f"{info['total_episodes']:,}")
print("Frames    :", f"{info['total_frames']:,}")
print("FPS       :", info["fps"])
print("Robot     :", info.get("robot_type"))
print("PASS: π0.5 preflight without JAX/GPU")


Episodes  : 1,000
Frames    : 699,921
FPS       : 20
Robot     : piper_bridge
PASS: π0.5 preflight without JAX/GPU


## 3. π0.5 quantile normalization asset 준비

π0와 π0.5는 같은 raw state/action에서 통계를 계산하지만 사용 방식이 다르다.

- π0: `mean/std` z-score
- π0.5: `q01/q99` quantile normalization

같은 dataset에서 계산된 현재 통계에는 네 필드가 모두 있으므로 byte-identical copy를 별도
`pi05_piper_lora` namespace에 설치할 수 있다. 기본값은 쓰기 비활성이다.


In [3]:
# Cell 3 — 통계 구조 검증과 선택적 π0.5 asset 설치
import hashlib
import shutil
import tempfile


def sha256_file(path: Path) -> str:
    """파일 SHA256을 반환한다."""

    return hashlib.sha256(path.read_bytes()).hexdigest()


raw_norm = json.loads(PI0_NORM_PATH.read_text(encoding="utf-8"))
assert set(raw_norm) == {"norm_stats"}
assert set(raw_norm["norm_stats"]) == {"state", "actions"}
for key in ("state", "actions"):
    assert set(raw_norm["norm_stats"][key]) == {"mean", "std", "q01", "q99"}
    for field in ("mean", "std", "q01", "q99"):
        assert len(raw_norm["norm_stats"][key][field]) == 7

PREPARE_PI05_ASSET = False
if PREPARE_PI05_ASSET:
    PI05_NORM_PATH.parent.mkdir(parents=True, exist_ok=True)
    if PI05_NORM_PATH.exists():
        if sha256_file(PI05_NORM_PATH) != sha256_file(PI0_NORM_PATH):
            raise FileExistsError(f"다른 π0.5 norm asset이 이미 있습니다: {PI05_NORM_PATH}")
    else:
        with tempfile.NamedTemporaryFile(dir=PI05_NORM_PATH.parent, delete=False) as temporary:
            temporary_path = Path(temporary.name)
        try:
            shutil.copyfile(PI0_NORM_PATH, temporary_path)
            temporary_path.replace(PI05_NORM_PATH)
        finally:
            temporary_path.unlink(missing_ok=True)
    print("Installed :", PI05_NORM_PATH)
else:
    print("검증만 완료했습니다. 설치하려면 PREPARE_PI05_ASSET=True로 바꿔 이 cell을 다시 실행하세요.")

print("SHA256    :", sha256_file(PI0_NORM_PATH))


검증만 완료했습니다. 설치하려면 PREPARE_PI05_ASSET=True로 바꿔 이 cell을 다시 실행하세요.
SHA256    : 813ef76b8f22595f73edfece605abf3861da3128325173b1b71d6aca778e3907


## 4. OpenPI π0.5 모델 계약

다음 cell부터 JAX/OpenPI를 import한다. 모델 weight는 아직 읽지 않지만 JAX backend가 초기화될 수 있다.


In [4]:
# Cell 4 — JAX/OpenPI API import와 공식 train module 연결
import dataclasses
import functools
import importlib.util
from typing import Any, Iterator, TypeAlias

import jax
import numpy as np
from flax import nnx
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from typing_extensions import override

from openpi import transforms
from openpi.models import model as model_api
from openpi.models import pi0_config
from openpi.policies import libero_policy
from openpi.shared import nnx_utils
from openpi.training import checkpoints, config as training_config, data_loader, optimizer as training_optimizer
from openpi.training import sharding as training_sharding
from openpi.training import weight_loaders
from piper_vla.common.dataset_v3 import PiperV3Dataset


def load_openpi_train_module() -> Any:
    """고정 submodule의 공식 train.py를 module로 불러온다."""

    path = OPENPI_ROOT / "scripts" / "train.py"
    spec = importlib.util.spec_from_file_location("pi05_notebook_openpi_train", path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"OpenPI train module을 읽지 못했습니다: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


OPENPI_TRAIN = load_openpi_train_module()
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())


/home/pc/vla_ws/.conda/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX backend: gpu
JAX devices: [CudaDevice(id=0)]


In [5]:
# Cell 5 — Piper π0.5 LoRA 모델과 freeze filter 확정
PIPER_ROBOT_DIM = 7
PIPER_DELTA_JOINT_DIM = 6
PI05_ACTION_HORIZON = 50
PI05_MODEL_ACTION_DIM = 32
PI05_IMAGE_SIZE = 224
PI05_PROMPT_TOKEN_LENGTH = 200
VISION_ENCODER_MODE = "frozen"  # "trainable" 또는 "frozen"

PI05_MODEL_CONFIG = pi0_config.Pi0Config(
    pi05=True,
    paligemma_variant="gemma_2b_lora",
    action_expert_variant="gemma_300m_lora",
    action_dim=PI05_MODEL_ACTION_DIM,
    action_horizon=PI05_ACTION_HORIZON,
    max_token_len=PI05_PROMPT_TOKEN_LENGTH,
    discrete_state_input=True,
)

if PI05_MODEL_CONFIG.model_type is not model_api.ModelType.PI05:
    raise ValueError(f"π0.5 model type이 아닙니다: {PI05_MODEL_CONFIG.model_type}")

official_lora_filter = PI05_MODEL_CONFIG.get_freeze_filter()
vision_filter = nnx_utils.PathRegex(r"PaliGemma/img/.*")
if VISION_ENCODER_MODE == "trainable":
    PI05_FREEZE_FILTER = official_lora_filter
elif VISION_ENCODER_MODE == "frozen":
    PI05_FREEZE_FILTER = nnx.Any(official_lora_filter, vision_filter)
else:
    raise ValueError(f"알 수 없는 Vision encoder mode: {VISION_ENCODER_MODE!r}")

print("Model type       :", PI05_MODEL_CONFIG.model_type)
print("LoRA variants    :", PI05_MODEL_CONFIG.paligemma_variant, "/", PI05_MODEL_CONFIG.action_expert_variant)
print("Action shape     :", (PI05_ACTION_HORIZON, PI05_MODEL_ACTION_DIM))
print("Prompt tokens    :", PI05_PROMPT_TOKEN_LENGTH)
print("Discrete state   :", PI05_MODEL_CONFIG.discrete_state_input)
print("Vision encoder   :", VISION_ENCODER_MODE)


Model type       : ModelType.PI05
LoRA variants    : gemma_2b_lora / gemma_300m_lora
Action shape     : (50, 32)
Prompt tokens    : 200
Discrete state   : True
Vision encoder   : frozen


## 5. Piper π0.5 data adapter prototype

π0 adapter를 그대로 import하지 않는다. π0.5 model type과 quantile normalization을 명시적으로 허용하고,
나머지 Piper 7D·두 카메라·delta joint 계약만 동일하게 유지한다.


In [6]:
# Cell 6 — π0.5 전용 DataConfigFactory와 local LeRobot v3 loader
PIPER_CANONICAL_REPACK = {
    "observation/image": "observation/image",
    "observation/wrist_image": "observation/wrist_image",
    "observation/state": "observation/state",
    "actions": "actions",
    "prompt": "prompt",
}
PIPER_DELTA_MASK = transforms.make_bool_mask(PIPER_DELTA_JOINT_DIM, -1)
Pi05TrainingBatch: TypeAlias = tuple[model_api.Observation, jax.Array]


def validate_pi05_norm_stats(norm_stats: dict[str, Any] | None) -> None:
    """π0.5 quantile normalization에 필요한 7D 통계를 검증한다."""

    if norm_stats is None:
        raise FileNotFoundError("π0.5 norm_stats.json을 불러오지 못했습니다.")
    if set(norm_stats) != {"state", "actions"}:
        raise ValueError(f"norm stats key 오류: {tuple(norm_stats)}")
    for key in ("state", "actions"):
        stats = norm_stats[key]
        for field in ("mean", "std", "q01", "q99"):
            values = np.asarray(getattr(stats, field, None))
            if values.shape != (PIPER_ROBOT_DIM,) or not np.isfinite(values).all():
                raise ValueError(f"{key}.{field} 계약 오류: shape={values.shape}")
        if np.any(np.asarray(stats.std) <= 0):
            raise ValueError(f"{key}.std는 모두 양수여야 합니다.")
        if np.any(np.asarray(stats.q01) > np.asarray(stats.q99)):
            raise ValueError(f"{key} quantile 순서가 잘못됐습니다.")


@dataclasses.dataclass(frozen=True)
class PiperPi05DataConfigFactory(training_config.DataConfigFactory):
    """Piper v3 sample을 π0.5 quantile/discrete-state transform으로 연결한다."""

    default_prompt: str | None = None

    @override
    def create(self, assets_dirs: Path, model_config: model_api.BaseModelConfig) -> training_config.DataConfig:
        if model_config.model_type is not model_api.ModelType.PI05:
            raise ValueError(f"π0.5가 아닌 model type입니다: {model_config.model_type}")
        if model_config.action_horizon != PI05_ACTION_HORIZON or model_config.action_dim != PI05_MODEL_ACTION_DIM:
            raise ValueError("Piper π0.5 action horizon/dimension 계약이 다릅니다.")

        base = self.create_base_config(assets_dirs, model_config)
        validate_pi05_norm_stats(base.norm_stats)
        if not base.use_quantile_norm:
            raise ValueError("π0.5는 quantile normalization을 사용해야 합니다.")

        return dataclasses.replace(
            base,
            repack_transforms=transforms.Group(
                inputs=(transforms.RepackTransform(PIPER_CANONICAL_REPACK),),
            ),
            data_transforms=transforms.Group(
                inputs=(
                    libero_policy.LiberoInputs(model_type=model_config.model_type),
                    transforms.DeltaActions(PIPER_DELTA_MASK),
                ),
                outputs=(
                    transforms.AbsoluteActions(PIPER_DELTA_MASK),
                    libero_policy.LiberoOutputs(),
                ),
            ),
            model_transforms=training_config.ModelTransformFactory(
                default_prompt=self.default_prompt,
            )(model_config),
            use_quantile_norm=True,
            action_sequence_keys=("actions",),
            prompt_from_task=False,
        )


@dataclasses.dataclass(frozen=True)
class Pi05DataBundle:
    """π0.5 smoke에 필요한 dataset, loader와 JAX sharding을 묶는다."""

    raw_dataset: PiperV3Dataset
    data_config: training_config.DataConfig
    loader: data_loader.DataLoaderImpl
    iterator: Iterator[Pi05TrainingBatch]
    mesh: Mesh
    data_sharding: NamedSharding
    replicated_sharding: NamedSharding


def build_pi05_data_bundle(config: training_config.TrainConfig) -> Pi05DataBundle:
    """local Piper dataset 전체를 OpenPI π0.5 JAX loader에 연결한다."""

    raw_dataset = PiperV3Dataset(
        root=DATASET_ROOT,
        action_horizon=config.model.action_horizon,
        validate_samples=False,
    )
    data_config = config.data.create(config.assets_dirs, config.model)
    validate_pi05_norm_stats(data_config.norm_stats)
    if not data_config.use_quantile_norm:
        raise ValueError("loader에 quantile normalization이 적용되지 않았습니다.")

    transformed = data_loader.transform_dataset(raw_dataset, data_config, skip_norm_stats=False)
    mesh = training_sharding.make_mesh(config.fsdp_devices)
    data_sharding = NamedSharding(mesh, P(training_sharding.DATA_AXIS))
    replicated_sharding = NamedSharding(mesh, P())
    inner = data_loader.TorchDataLoader(
        transformed,
        local_batch_size=config.batch_size // jax.process_count(),
        sharding=data_sharding,
        shuffle=True,
        sampler=None,
        num_batches=None,
        num_workers=config.num_workers,
        seed=config.seed,
        framework="jax",
    )
    loader = data_loader.DataLoaderImpl(data_config, inner)
    return Pi05DataBundle(
        raw_dataset=raw_dataset,
        data_config=data_config,
        loader=loader,
        iterator=iter(loader),
        mesh=mesh,
        data_sharding=data_sharding,
        replicated_sharding=replicated_sharding,
    )


def validate_pi05_batch(batch: Pi05TrainingBatch, config: training_config.TrainConfig) -> None:
    """첫 π0.5 batch의 image/state/action/token shape와 유한값을 확인한다."""

    observation, actions = batch
    batch_size = config.batch_size
    assert observation.state.shape == (batch_size, PI05_MODEL_ACTION_DIM)
    assert actions.shape == (batch_size, PI05_ACTION_HORIZON, PI05_MODEL_ACTION_DIM)
    assert observation.tokenized_prompt.shape == (batch_size, PI05_PROMPT_TOKEN_LENGTH)
    assert observation.tokenized_prompt_mask.shape == (batch_size, PI05_PROMPT_TOKEN_LENGTH)
    assert np.isfinite(np.asarray(jax.device_get(observation.state))).all()
    assert np.isfinite(np.asarray(jax.device_get(actions))).all()
    assert np.allclose(np.asarray(jax.device_get(observation.state))[..., PIPER_ROBOT_DIM:], 0.0)
    assert np.allclose(np.asarray(jax.device_get(actions))[..., PIPER_ROBOT_DIM:], 0.0)


## 6. 실험용 π0.5 LoRA TrainConfig

다음 값은 공식 Piper π0.5 추천값이 아니다. 먼저 1-step smoke를 통과시키기 위한 보수적인 시작점이다.

- batch 1, worker 0
- 1 step
- AdamW와 cosine schedule은 기존 π0 LoRA baseline
- EMA 없음
- π0.5 base weight에서 fresh 시작


In [7]:
# Cell 7 — 별도 asset/run namespace를 쓰는 π0.5 LoRA config
PI05_TRAIN_CONFIG = training_config.TrainConfig(
    name=PI05_CONFIG_NAME,
    exp_name=RUN_NAME,
    model=PI05_MODEL_CONFIG,
    data=PiperPi05DataConfigFactory(
        repo_id=ASSET_ID,
        assets=training_config.AssetsConfig(asset_id=ASSET_ID),
        default_prompt=DEFAULT_PROMPT,
    ),
    weight_loader=weight_loaders.CheckpointWeightLoader(str(PI05_BASE_PARAMS)),
    lr_schedule=training_optimizer.CosineDecaySchedule(
        warmup_steps=1_000,
        peak_lr=2.5e-5,
        decay_steps=30_000,
        decay_lr=2.5e-6,
    ),
    optimizer=training_optimizer.AdamW(
        b1=0.9,
        b2=0.95,
        eps=1e-8,
        weight_decay=1e-10,
        clip_gradient_norm=1.0,
    ),
    ema_decay=None,
    freeze_filter=PI05_FREEZE_FILTER,
    assets_base_dir=str(WORKSPACE_ROOT / "data" / "assets"),
    checkpoint_base_dir=str(RUNS_ROOT),
    batch_size=1,
    num_workers=0,
    num_train_steps=1,
    log_interval=1,
    save_interval=1,
    keep_period=1,
    seed=42,
    fsdp_devices=1,
    wandb_enabled=False,
    overwrite=False,
    resume=False,
)

assert PI05_TRAIN_CONFIG.model.model_type is model_api.ModelType.PI05
assert PI05_TRAIN_CONFIG.model.pi05 is True
assert PI05_TRAIN_CONFIG.model.discrete_state_input is True
assert PI05_TRAIN_CONFIG.assets_dirs == (WORKSPACE_ROOT / "data" / "assets" / PI05_CONFIG_NAME).resolve()
assert PI05_TRAIN_CONFIG.checkpoint_dir == (RUNS_ROOT / PI05_CONFIG_NAME / RUN_NAME).resolve()

print("Config      :", PI05_TRAIN_CONFIG.name)
print("Checkpoint  :", PI05_TRAIN_CONFIG.checkpoint_dir)
print("Norm asset  :", PI05_NORM_PATH)
print("Batch/steps :", PI05_TRAIN_CONFIG.batch_size, "/", PI05_TRAIN_CONFIG.num_train_steps)
print("WARNING: optimizer/LR은 실험 baseline이며 공식 Piper π0.5 권장값이 아닙니다.")


Config      : pi05_piper_lora
Checkpoint  : /home/pc/vla_ws/data/runs/pi05_piper_lora/two_block_pnp_b1_pi05_vf_s1_smoke_r001
Norm asset  : /home/pc/vla_ws/data/assets/pi05_piper_lora/two_block_pnp/norm_stats.json
Batch/steps : 1 / 1


## 7. 첫 π0.5 data batch 검사

Cell 3에서 `PREPARE_PI05_ASSET=True`로 통계를 설치한 뒤 실행한다. model weight는 아직 읽지 않는다.


In [8]:
# Cell 8 — quantile/discrete-state transform을 거친 실제 첫 batch
if not PI05_NORM_PATH.is_file():
    raise FileNotFoundError(
        "π0.5 norm asset이 없습니다. Cell 3의 PREPARE_PI05_ASSET=True를 실행하세요: "
        f"{PI05_NORM_PATH}"
    )

PI05_DATA = build_pi05_data_bundle(PI05_TRAIN_CONFIG)
FIRST_BATCH = next(PI05_DATA.iterator)
validate_pi05_batch(FIRST_BATCH, PI05_TRAIN_CONFIG)

observation, actions = FIRST_BATCH
print("Dataset frames :", f"{len(PI05_DATA.raw_dataset):,}")
print("State          :", observation.state.shape, observation.state.dtype)
print("Actions        :", actions.shape, actions.dtype)
print("Prompt tokens  :", observation.tokenized_prompt.shape)
print("Quantile norm  :", PI05_DATA.data_config.use_quantile_norm)
print("PASS: Piper v3 -> OpenPI π0.5 data contract")


Dataset frames : 699,921
State          : (1, 32) float32
Actions        : (1, 50, 32) float32
Prompt tokens  : (1, 200)
Quantile norm  : True
PASS: Piper v3 -> OpenPI π0.5 data contract


## 8. abstract TrainState와 parameter partition

weight를 읽기 전에 모델 shape와 optimizer state를 추상적으로 만든다. 결과에서 전체·학습 가능 parameter 수를
확인한다. π0 수치를 그대로 기대하면 안 되며, 여기서 얻은 π0.5 수치를 production 회귀 테스트에 고정한다.


In [9]:
# Cell 9 — base weight를 읽지 않는 π0.5 abstract state audit
ABSTRACT_STATE, PI05_STATE_SHARDING = OPENPI_TRAIN.init_train_state(
    PI05_TRAIN_CONFIG,
    jax.random.key(PI05_TRAIN_CONFIG.seed),
    PI05_DATA.mesh,
    resume=True,
)

total_params = sum(int(leaf.size) for leaf in jax.tree.leaves(ABSTRACT_STATE.params))
trainable_params = sum(
    int(leaf.size)
    for leaf in jax.tree.leaves(ABSTRACT_STATE.params.filter(PI05_TRAIN_CONFIG.trainable_filter))
)
frozen_params = total_params - trainable_params

if total_params != trainable_params + frozen_params or trainable_params <= 0:
    raise RuntimeError("π0.5 parameter partition이 올바르지 않습니다.")

print("Total params     :", f"{total_params:,}")
print("Trainable params :", f"{trainable_params:,}", f"({100 * trainable_params / total_params:.3f}%)")
print("Frozen params    :", f"{frozen_params:,}")
print("Vision mode      :", VISION_ENCODER_MODE)


Total params     : 3,403,421,456
Trainable params : 52,153,376 (1.532%)
Frozen params    : 3,351,268,080
Vision mode      : frozen


## 9. π0.5 base weight 로드

이 cell부터 실제 GPU memory를 사용한다. 먼저 `nvidia-smi`에서 다른 JAX/LLM process가 없는지 확인한다.
기본값 `False`에서는 아무것도 로드하지 않는다. 새 run 경로가 이미 있으면 자동 overwrite하지 않는다.


In [11]:
# Cell 10 — 명시적으로 허용했을 때만 pi05_base weight와 LoRA TrainState 로드
ENABLE_GPU_WEIGHT_LOAD = True

if not ENABLE_GPU_WEIGHT_LOAD:
    print("SKIP: ENABLE_GPU_WEIGHT_LOAD=True로 바꾼 뒤 다시 실행하세요.")
else:
    if jax.default_backend() != "gpu":
        raise RuntimeError(f"GPU backend가 아닙니다: {jax.default_backend()}")
    if PI05_TRAIN_CONFIG.checkpoint_dir.exists():
        raise FileExistsError(
            "fresh smoke run 경로가 이미 있습니다. RUN_NAME을 바꾸세요: "
            f"{PI05_TRAIN_CONFIG.checkpoint_dir}"
        )

    master_rng = jax.random.key(PI05_TRAIN_CONFIG.seed)
    PI05_TRAIN_RNG, init_rng = jax.random.split(master_rng)
    PI05_STATE, PI05_STATE_SHARDING = OPENPI_TRAIN.init_train_state(
        PI05_TRAIN_CONFIG,
        init_rng,
        PI05_DATA.mesh,
        resume=False,
    )
    PI05_STATE = jax.block_until_ready(PI05_STATE)
    assert int(jax.device_get(PI05_STATE.step)) == 0
    print("PASS: pi05_base -> π0.5 LoRA TrainState step 0")


PASS: pi05_base -> π0.5 LoRA TrainState step 0


## 10. 첫 JIT 학습 step

Cell 10이 실제로 실행된 경우에만 한 step을 수행한다. loss, grad norm, parameter norm은 모두 유한해야 한다.


In [12]:
# Cell 11 — 공식 OpenPI train_step으로 정확히 한 번 update
if "PI05_STATE" not in globals():
    print("SKIP: Cell 10에서 GPU weight load를 먼저 실행하세요.")
else:
    PI05_TRAIN_STEP = jax.jit(
        functools.partial(OPENPI_TRAIN.train_step, PI05_TRAIN_CONFIG),
        in_shardings=(
            PI05_DATA.replicated_sharding,
            PI05_STATE_SHARDING,
            PI05_DATA.data_sharding,
        ),
        out_shardings=(
            PI05_STATE_SHARDING,
            PI05_DATA.replicated_sharding,
        ),
        donate_argnums=(1,),
    )
    PI05_STATE, PI05_METRICS = PI05_TRAIN_STEP(
        PI05_TRAIN_RNG,
        PI05_STATE,
        FIRST_BATCH,
    )
    PI05_STATE, PI05_METRICS = jax.block_until_ready((PI05_STATE, PI05_METRICS))
    host_metrics = {key: float(value) for key, value in jax.device_get(PI05_METRICS).items()}
    if not all(np.isfinite(value) for value in host_metrics.values()):
        raise FloatingPointError(f"NaN/Inf metric: {host_metrics}")
    assert int(jax.device_get(PI05_STATE.step)) == 1
    print("Step    :", int(jax.device_get(PI05_STATE.step)))
    print("Metrics :", host_metrics)
    print("PASS: first π0.5 LoRA update")


Step    : 1
Metrics : {'grad_norm': 0.3728647232055664, 'loss': 0.06669788807630539, 'param_norm': 1803.7701416015625}
PASS: first π0.5 LoRA update


## 11. Step 1 checkpoint 저장

`params`, `train_state`, `assets/<asset_id>/norm_stats.json`을 저장하고 Orbax commit 완료를 확인한다.
이 cell은 약 9GB 이상의 디스크와 checkpoint serialization RAM을 사용할 수 있다.


In [13]:
# Cell 12 — step 1 checkpoint를 새 π0.5 namespace에 저장
if "PI05_STATE" not in globals() or int(jax.device_get(PI05_STATE.step)) != 1:
    print("SKIP: Cell 11의 step 1 state가 필요합니다.")
else:
    PI05_MANAGER, resuming = checkpoints.initialize_checkpoint_dir(
        PI05_TRAIN_CONFIG.checkpoint_dir,
        keep_period=PI05_TRAIN_CONFIG.keep_period,
        overwrite=False,
        resume=False,
    )
    if resuming:
        raise RuntimeError("fresh manager가 resume 상태로 열렸습니다.")
    checkpoints.save_state(PI05_MANAGER, PI05_STATE, PI05_DATA.loader, step=1)
    PI05_MANAGER.wait_until_finished()
    if PI05_MANAGER.latest_step() != 1:
        raise RuntimeError(f"step 1 commit 실패: {PI05_MANAGER.all_steps()}")
    embedded_norm = PI05_TRAIN_CONFIG.checkpoint_dir / "1" / "assets" / ASSET_ID / "norm_stats.json"
    if not embedded_norm.is_file():
        raise FileNotFoundError(f"checkpoint norm asset이 없습니다: {embedded_norm}")
    print("Checkpoint :", PI05_TRAIN_CONFIG.checkpoint_dir / "1")
    print("Norm asset :", embedded_norm)
    print("PASS: durable step 1 checkpoint")


Checkpoint : /home/pc/vla_ws/data/runs/pi05_piper_lora/two_block_pnp_b1_pi05_vf_s1_smoke_r001/1
Norm asset : /home/pc/vla_ws/data/runs/pi05_piper_lora/two_block_pnp_b1_pi05_vf_s1_smoke_r001/1/assets/two_block_pnp/norm_stats.json
PASS: durable step 1 checkpoint


## 12. Step 1 restore 계약

아래 cell은 같은 kernel에서 구조만 확인한다. production 구현에서는 새 process를 시작해 base weight를 다시
읽지 않고 `train_state + params`를 복원하는 별도 resume smoke가 필요하다.


In [ ]:
# Cell 13 — abstract state에 step 1 checkpoint 복원
if "PI05_MANAGER" not in globals() or PI05_MANAGER.latest_step() != 1:
    print("SKIP: Cell 12에서 checkpoint를 먼저 저장하세요.")
else:
    resume_config = dataclasses.replace(PI05_TRAIN_CONFIG, resume=True)
    resume_shape, resume_sharding = OPENPI_TRAIN.init_train_state(
        resume_config,
        jax.random.key(resume_config.seed),
        PI05_DATA.mesh,
        resume=True,
    )
    restored_state = checkpoints.restore_state(
        PI05_MANAGER,
        resume_shape,
        PI05_DATA.loader,
        step=1,
    )
    restored_state = jax.block_until_ready(restored_state)
    assert int(jax.device_get(restored_state.step)) == 1
    print("PASS: restored π0.5 TrainState step", int(jax.device_get(restored_state.step)))


/home/pc/vla_ws/.conda/env/lib/python3.11/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1251: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


PASS: restored π0.5 TrainState step 1


: 

## 13. Production 코드로 옮기기 전 통과 조건

다음 항목이 모두 통과해야 `src/piper_vla/training/pi05/` 구현을 시작한다.

1. quantile normalization과 200-token discrete-state batch
2. π0.5 LoRA parameter partition 수치 고정
3. `pi05_base`에서 fresh step 0 초기화
4. finite 1-step update
5. step 1 checkpoint 저장과 base weight 없는 restore

그 다음에만 batch `1 → 4 → 8` smoke를 수행한다. π0 checkpoint와 교차 resume하지 않는다.
